# Union-Closed Sets Conjecture: Interactive Exploration

Explore the conjecture using Quantum, Tensor Network, and GNN approaches.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from core.family import UnionClosedFamily
from core.generator import FamilyGenerator, generate_test_suite
from core.verifier import ConjectureVerifier

from approaches.quantum.density_matrix import QuantumApproach
from approaches.tensor.mps import TensorNetworkApproach

%matplotlib inline
sns.set_style('whitegrid')

## 1. Basic Examples

In [ ]:
# Example 1: Simple family
family1 = UnionClosedFamily([{1}, {2}, {1, 2}])

print("Family:", [set(s) for s in family1.sets])
print("Is union-closed:", family1.is_union_closed())
print("Frequencies:", family1.compute_frequencies())
print("Satisfies conjecture:", family1.satisfies_conjecture())
print("Min frequency:", family1.min_frequency())
print("Max frequency:", family1.max_frequency())

## 2. Quantum Approach

In [ ]:
# Analyze with quantum approach
qa = QuantumApproach(family1)

print("Quantum Statistics:")
stats = qa.quantum_statistics()
for key, value in stats.items():
    if not isinstance(value, (list, dict)):
        print(f"  {key}: {value}")

# Visualize density matrix
rho = qa.compute_density_matrix()
plt.figure(figsize=(8, 6))
plt.imshow(np.real(rho), cmap='viridis')
plt.colorbar(label='Density')
plt.title('Density Matrix')
plt.xlabel('Element')
plt.ylabel('Element')
plt.show()

# Eigenspectrum
eigenvalues = qa.eigenspectrum()
plt.figure(figsize=(8, 4))
plt.bar(range(len(eigenvalues)), eigenvalues)
plt.xlabel('Index')
plt.ylabel('Eigenvalue')
plt.title('Eigenspectrum of Density Matrix')
plt.show()

## 3. Tensor Network Approach

In [ ]:
# Analyze with tensor network
tna = TensorNetworkApproach(family1)

print("Tensor Statistics:")
tstats = tna.tensor_statistics()
for key, value in tstats.items():
    if not isinstance(value, (list, dict, np.ndarray)):
        print(f"  {key}: {value}")

# Entanglement profile
if tstats['entanglement_profile']:
    plt.figure(figsize=(8, 4))
    plt.plot(tstats['entanglement_profile'], marker='o')
    plt.xlabel('Cut Position')
    plt.ylabel('Entanglement Entropy')
    plt.title('Entanglement Profile')
    plt.grid(True)
    plt.show()

# Bond dimensions
if tstats['bond_dimensions']:
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(tstats['bond_dimensions'])), tstats['bond_dimensions'])
    plt.xlabel('Cut Position')
    plt.ylabel('Bond Dimension')
    plt.title('Bond Dimensions')
    plt.show()

## 4. Batch Analysis

In [ ]:
# Generate test families
num_test = 100
families = generate_test_suite(num_test, max_n=8, seed=42)

print(f"Generated {len(families)} test families")

# Analyze all
quantum_bounds = []
actual_freqs = []
entropies = []

for family in families:
    if family.n == 0 or family.m == 0:
        continue
    
    _, max_freq = family.max_frequency()
    actual_freqs.append(max_freq)
    
    try:
        qa = QuantumApproach(family)
        quantum_bounds.append(qa.quantum_frequency_bound())
        entropies.append(qa.von_neumann_entropy())
    except:
        pass

print(f"Successfully analyzed {len(quantum_bounds)} families")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of actual max frequencies
axes[0, 0].hist(actual_freqs, bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(0.5, color='red', linestyle='--', label='Conjecture threshold')
axes[0, 0].set_xlabel('Max Frequency')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Distribution of Max Frequencies')
axes[0, 0].legend()

# Quantum bound vs actual
axes[0, 1].scatter(quantum_bounds, actual_freqs, alpha=0.5)
axes[0, 1].plot([0, 1], [0, 1], 'r--', label='Perfect match')
axes[0, 1].axhline(0.5, color='orange', linestyle='--', alpha=0.5)
axes[0, 1].axvline(0.5, color='orange', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Quantum Bound')
axes[0, 1].set_ylabel('Actual Max Frequency')
axes[0, 1].set_title('Quantum Bound vs Actual')
axes[0, 1].legend()

# Von Neumann entropy distribution
axes[1, 0].hist(entropies, bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_xlabel('Von Neumann Entropy')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Distribution of Entropies')

# Entropy vs max frequency
axes[1, 1].scatter(entropies, actual_freqs, alpha=0.5, color='purple')
axes[1, 1].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Von Neumann Entropy')
axes[1, 1].set_ylabel('Max Frequency')
axes[1, 1].set_title('Entropy vs Max Frequency')

plt.tight_layout()
plt.show()

## 5. Search for Challenging Cases

In [ ]:
# Find families with lowest max frequency (but still >= 0.5)
challenging = []

for i, family in enumerate(families):
    if family.n == 0 or family.m == 0:
        continue
    
    if family.is_union_closed() and family.satisfies_conjecture():
        _, max_freq = family.max_frequency()
        if 0.5 <= max_freq <= 0.55:
            challenging.append((family, max_freq))

challenging = sorted(challenging, key=lambda x: x[1])

print(f"Found {len(challenging)} challenging cases")
print("\nTop 5 hardest:")
for i, (family, max_freq) in enumerate(challenging[:5]):
    print(f"{i+1}. Max freq: {max_freq:.4f}, n={family.n}, m={family.m}")

## 6. Deep Dive into One Challenging Case

In [ ]:
if challenging:
    hard_family, hard_freq = challenging[0]
    
    print("Challenging Family:")
    print(f"  Sets: {[set(s) for s in hard_family.sets]}")
    print(f"  Max frequency: {hard_freq:.4f}")
    print(f"  Frequencies: {hard_family.compute_frequencies()}")
    
    # Quantum analysis
    qa_hard = QuantumApproach(hard_family)
    q_stats = qa_hard.quantum_statistics()
    print(f"\n  Quantum bound: {q_stats['quantum_bound']:.4f}")
    print(f"  Von Neumann entropy: {q_stats['von_neumann_entropy']:.4f}")
    print(f"  Purity: {q_stats['purity']:.4f}")
    
    # Tensor analysis
    tna_hard = TensorNetworkApproach(hard_family)
    t_stats = tna_hard.tensor_statistics()
    print(f"\n  Tensor rank: {t_stats['tensor_rank']}")
    print(f"  Avg entanglement: {t_stats['avg_entanglement']:.4f}")
    print(f"  Max bond dimension: {t_stats['max_bond_dimension']}")

## 7. Conjecture Verification Statistics

In [ ]:
# Verify conjecture on all families
verifier = ConjectureVerifier()
verification = verifier.verify_batch(families)

print("Verification Results:")
print(f"  Total families: {verification['total_families']}")
print(f"  Union-closed: {verification['union_closed_count']}")
print(f"  Satisfying conjecture: {verification['satisfies_count']}")
print(f"  Counterexamples: {verification['counterexample_count']}")
print(f"  Avg min frequency: {verification['avg_min_frequency']:.4f}")
print(f"  Avg max frequency: {verification['avg_max_frequency']:.4f}")

if verification['counterexamples']:
    print("\n⚠️ COUNTEREXAMPLES FOUND! ⚠️")
    for ce in verification['counterexamples']:
        print(ce)
else:
    print("\n✅ No counterexamples found in this batch!")

## 8. Next Steps

To continue exploration:
1. Train GNN model: `python approaches/gnn/train.py`
2. Run full batch analysis: `python run_all.py --mode batch --num-families 1000`
3. Search for challenging cases: `python run_all.py --mode challenging --num-families 10000`
4. Compare approaches: `python run_all.py --mode compare`